# Notebook 5: Deeper Networks
**Estimated time:** 35-45 min
**Prerequisites:** Notebooks 1-4

---
## What you will learn
1. Why deep networks are more powerful — but also trickier
2. Activation functions in depth (with plots and intuition)
3. The vanishing gradient problem
4. Batch Normalization — what it does and why it helps
5. Dropout — regularization by random deactivation
6. How to structure a deep network properly

## 1. Why Go Deep?

**Shallow networks** can approximate any function (Universal Approximation Theorem) — but they need exponentially many neurons.

**Deep networks** learn *hierarchical representations*:
- Layer 1: edges and corners
- Layer 2: shapes and textures
- Layer 3: object parts
- Layer 4: full objects

This is how your brain works too. You don't recognize faces by checking every pixel combination — you first detect edges, then facial features, then the full face.

The challenge: as networks get deeper, training becomes unstable.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_circles
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

torch.manual_seed(42)

## 2. The Vanishing Gradient Problem

**Imagine passing a whisper through 100 people in a chain.**
Each person speaks slightly more quietly. By the end, the original message is gone.

In deep networks, gradients flow backward through layers. With sigmoid/tanh activations:
- Gradients get multiplied by a value between 0 and 0.25 at each layer
- After 10 layers: 0.25^10 ≈ 0.000001 — practically zero
- Early layers receive near-zero gradients → they barely learn

**Solutions:**
1. Use **ReLU** activations (gradient is 0 or 1, not squashed)
2. Use **Batch Normalization** (normalizes activations between layers)
3. Use **residual connections** (skip connections — covered in CNNs)

In [ ]:
# Demonstrate vanishing gradients with sigmoid vs ReLU
def check_gradients(activation_name, activation_fn, depth=10):
    x = torch.randn(64, 10, requires_grad=False)
    layers = nn.Sequential(*[
        nn.Sequential(nn.Linear(10, 10), activation_fn)
        for _ in range(depth)
    ])

    x_in = torch.randn(64, 10, requires_grad=True)
    out = layers(x_in)
    loss = out.sum()
    loss.backward()

    # Check gradient magnitude at input
    grad_norm = x_in.grad.abs().mean().item()
    return grad_norm

sigmoid_grad = check_gradients('Sigmoid', nn.Sigmoid())
tanh_grad    = check_gradients('Tanh',    nn.Tanh())
relu_grad    = check_gradients('ReLU',    nn.ReLU())

print(f'Gradient norm after 10 layers:')
print(f'  Sigmoid: {sigmoid_grad:.8f}  (vanished!)')
print(f'  Tanh   : {tanh_grad:.8f}  (vanished!)')
print(f'  ReLU   : {relu_grad:.8f}  (healthy)')

## 3. Activation Functions — Which to Use?

| Activation | Use case |
|------------|----------|
| `ReLU` | Default for hidden layers in feedforward/CNNs |
| `LeakyReLU` | When you suspect dead neurons (ReLU never fires) |
| `GELU` | Transformers, BERT, GPT — smooth version of ReLU |
| `Sigmoid` | Binary output layer (probability between 0 and 1) |
| `Softmax` | Multi-class output layer (probabilities sum to 1) |
| `Tanh` | Hidden layers in RNNs |

**Dead ReLU problem:** If a neuron's weights are updated so it always gets negative input, ReLU outputs 0 forever and the gradient is 0 → neuron is "dead". LeakyReLU fixes this by having a small slope for negatives.

In [ ]:
x = torch.linspace(-4, 4, 200)

fns = {
    'ReLU'        : nn.ReLU(),
    'LeakyReLU(0.2)': nn.LeakyReLU(0.2),
    'GELU'        : nn.GELU(),
    'Sigmoid'     : nn.Sigmoid(),
    'Tanh'        : nn.Tanh(),
    'Softplus'    : nn.Softplus(),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (name, fn) in zip(axes.flat, fns.items()):
    with torch.no_grad():
        y = fn(x)
    ax.plot(x.numpy(), y.numpy(), linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(name, fontsize=13)
    ax.set_ylim(-1.5, 4.5)
    ax.grid(True, alpha=0.3)
plt.suptitle('Activation Functions', fontsize=15)
plt.tight_layout()
plt.show()

## 4. Batch Normalization

**Imagine you're baking cakes. Each layer of the cake needs the right amount of batter.**
If one layer is too thick, it affects all layers above. BatchNorm re-centers and re-scales each layer's output so the next layer always gets a well-behaved input.

Formally, for a batch of activations:
```
x_norm = (x - mean(x)) / sqrt(var(x) + eps)
output = gamma * x_norm + beta   # learnable scale and shift
```

**Benefits:**
- Reduces internal covariate shift (inputs to each layer stay stable)
- Allows higher learning rates
- Acts as a mild regularizer
- Dramatically helps with very deep networks

**Rule of thumb:** Put BatchNorm after the linear layer and before the activation.

In [ ]:
# BatchNorm1d for 1D (tabular) features
# BatchNorm2d for 2D (image) feature maps

class DeepNetWithBN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, depth=5):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU()]
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class DeepNetNoBN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, depth=5):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model_bn  = DeepNetWithBN(2, 64, 2, depth=8)
model_nbn = DeepNetNoBN(2,  64, 2, depth=8)
print('Model WITH BatchNorm:', sum(p.numel() for p in model_bn.parameters()), 'params')
print('Model WITHOUT BatchNorm:', sum(p.numel() for p in model_nbn.parameters()), 'params')

In [ ]:
# Train both models and compare
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=2000, noise=0.3, random_state=0)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)
dataset = TensorDataset(X, y)
loader  = DataLoader(dataset, batch_size=64, shuffle=True)
criterion = nn.CrossEntropyLoss()

def train_model(model, epochs=80):
    opt = optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_b, y_b in loader:
            opt.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))
    return losses

losses_bn  = train_model(model_bn)
losses_nbn = train_model(model_nbn)

plt.figure(figsize=(8, 4))
plt.plot(losses_bn,  label='With BatchNorm')
plt.plot(losses_nbn, label='Without BatchNorm')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('BatchNorm makes training more stable in deep networks')
plt.legend()
plt.show()

## 5. Dropout

**Imagine studying for an exam, but every day you randomly forget some of what you learned.**
You're forced to learn the material multiple ways, not rely on any one path.
When exam day comes (no forgetting), you do better because you built redundant knowledge.

Dropout randomly sets a fraction `p` of neurons to 0 during training. Each forward pass is a different random sub-network.

**Benefits:**
- Forces the network to not rely on any single neuron
- Equivalent to training an ensemble of many networks
- Prevents overfitting

**Important:** Dropout is **only active during `model.train()`**. During `model.eval()`, all neurons are active (outputs are scaled to compensate).

In [ ]:
class NetworkWithDropout(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),       # drop 30% of neurons
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)


model = NetworkWithDropout(2, 64, 2, dropout_p=0.3)

# Demonstrate that dropout is inactive during eval
x = torch.randn(1, 2)

model.train()
out_train1 = model(x)
out_train2 = model(x)  # different! (different neurons dropped)

model.eval()
out_eval1 = model(x)
out_eval2 = model(x)   # same! (no dropout)

print('TRAIN mode (different each forward pass):')
print(' ', out_train1.detach())
print(' ', out_train2.detach())
print('EVAL mode (deterministic):')
print(' ', out_eval1.detach())
print(' ', out_eval2.detach())

## 6. Best Practices for Deep Networks

A well-structured deep network typically follows this pattern per layer:
```
Linear → BatchNorm → Activation → Dropout (optional)
```

Summary of what we've learned:
- Use **ReLU** (or GELU for modern architectures)
- Add **BatchNorm** after linear layers in deep networks
- Add **Dropout** to reduce overfitting (typical values: 0.1 – 0.5)
- Check gradient magnitudes if training is unstable
- Start shallow and add depth only when needed

---
## YOUR TURN — Mini Project

**Task:** Compare 4 architectures on the `make_moons` dataset:

| Architecture | Layers | BatchNorm | Dropout |
|-------------|--------|-----------|---------|
| A | 3 linear (32 hidden) | No | No |
| B | 3 linear (32 hidden) | Yes | No |
| C | 3 linear (32 hidden) | No | Yes (p=0.3) |
| D | 3 linear (32 hidden) | Yes | Yes (p=0.3) |

**Steps:**
1. Generate 2000 samples (`noise=0.2`), split 80/20 train/val
2. Train each for 100 epochs with Adam
3. Plot all 4 training loss curves on one graph
4. Print final validation accuracy for each
5. Which performs best? Why do you think?

**Hint:** Use `make_moons` from sklearn, wrap in `TensorDataset`, use `DataLoader`.

In [ ]:
# YOUR CODE HERE
# Define 4 model classes (or use nn.Sequential with conditional layers)
# Train and compare all 4